<a href="https://colab.research.google.com/github/Capechusami/Zindi_Financial_Inclusion/blob/0.099157164(submission_stack_country_threshold(0.099157164))/Zindi_Financial_Inclusion__2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# ── Install CatBoost if needed, then import everything ─────────────────────
try:
    from catboost import CatBoostClassifier
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'catboost', '-q'], check=True)
    from catboost import CatBoostClassifier

import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import time
import warnings
warnings.filterwarnings('ignore')

try:
    from lightgbm import early_stopping
    LGBM_NEW_API = True
except ImportError:
    LGBM_NEW_API = False

# ── Data path ──────────────────────────────────────────────────────────────
DATA_PATH = './'

In [13]:
# ── Load data ──────────────────────────────────────────────────────────────
train = pd.read_csv(DATA_PATH + 'Train.csv')
test  = pd.read_csv(DATA_PATH + 'Test.csv')

print('Train:', train.shape, ' | Test:', test.shape)
print('\nTarget distribution:')
print(train['bank_account'].value_counts())

# Save submission IDs before any transformation
test_ids = test['uniqueid'] + ' x ' + test['country']

# Encode target: Yes -> 1, No -> 0
train['bank_account'] = (train['bank_account'] == 'Yes').astype(int)
print('\nPositive rate:', round(train['bank_account'].mean(), 4))

Train: (23524, 13)  | Test: (10086, 12)

Target distribution:
bank_account
No     20212
Yes     3312
Name: count, dtype: int64

Positive rate: 0.1408


In [14]:
# ── Feature engineering (v3 — NO leaky target encoding here) ──────────────
EDU_ORDER = {
    'No formal education': 0,
    'Primary education': 1,
    'Secondary education': 2,
    'Vocational/Specialised training': 3,
    'Tertiary education': 4,
    'Other/Dont know/RTA': 2,
}

def engineer_features(df):
    df = df.copy()

    # Education ordinal
    df['education_ordinal'] = df['education_level'].map(EDU_ORDER).fillna(2).astype(int)

    # Age features
    df['log_age'] = np.log1p(df['age_of_respondent'])
    df['age_bin'] = pd.cut(df['age_of_respondent'],
                           bins=[0, 20, 30, 40, 50, 60, 100],
                           labels=False).fillna(0).astype(int)

    # Household features
    df['log_household'] = np.log1p(df['household_size'])
    df['household_bin'] = pd.cut(df['household_size'],
                                  bins=[0, 2, 4, 6, 8, 100],
                                  labels=False).fillna(0).astype(int)

    # Numeric interactions (no target leakage)
    df['age_x_edu']       = df['age_of_respondent'] * df['education_ordinal']
    df['household_x_edu'] = df['household_size']    * df['education_ordinal']
    df['age_x_household'] = df['age_of_respondent'] * df['household_size']

    return df

In [15]:
# ── Apply feature engineering (safe FE only) ───────────────────────────────
train = engineer_features(train)
test  = engineer_features(test)

# ── Categorical label encoding (fit on train + test combined) ──────────────
CAT_COLS = ['country', 'location_type', 'cellphone_access',
            'gender_of_respondent', 'relationship_with_head',
            'marital_status', 'education_level', 'job_type']

for col in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

# ── Define features ────────────────────────────────────────────────────────
DROP_COLS    = ['bank_account', 'uniqueid']
FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

X      = train[FEATURE_COLS].copy()
y      = train['bank_account'].copy()
X_test = test[FEATURE_COLS].copy()

print('Features before TE:', len(FEATURE_COLS))
print(FEATURE_COLS)

Features before TE: 19
['country', 'year', 'location_type', 'cellphone_access', 'household_size', 'age_of_respondent', 'gender_of_respondent', 'relationship_with_head', 'marital_status', 'education_level', 'job_type', 'education_ordinal', 'log_age', 'age_bin', 'log_household', 'household_bin', 'age_x_edu', 'household_x_edu', 'age_x_household']


In [16]:
# ── Proper Out-of-Fold Target Encoding (no leakage) ────────────────────────
def oof_target_encode(series_train, y_train, series_test, smoothing=30,
                      n_splits=5, seed=0):
    kf       = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof      = np.zeros(len(series_train))
    gm       = y_train.mean()

    series_train = series_train.reset_index(drop=True)
    y_train      = y_train.reset_index(drop=True)
    series_test  = series_test.reset_index(drop=True)

    for tr_idx, val_idx in kf.split(series_train):
        stats = pd.DataFrame({
            'c': series_train.iloc[tr_idx].values,
            'y': y_train.iloc[tr_idx].values,
        }).groupby('c')['y'].agg(['mean', 'count'])
        smoothed        = (stats['count'] * stats['mean'] +
                           smoothing * gm) / (stats['count'] + smoothing)
        oof[val_idx]    = series_train.iloc[val_idx].map(smoothed).fillna(gm).values

    stats = pd.DataFrame({
        'c': series_train.values,
        'y': y_train.values,
    }).groupby('c')['y'].agg(['mean', 'count'])
    smoothed = (stats['count'] * stats['mean'] + smoothing * gm) / (stats['count'] + smoothing)
    test_enc = series_test.map(smoothed).fillna(gm).values

    return oof, test_enc


# ── 1-way TE on high-cardinality columns only ─────────────────────────────
single_te_cols = ['country', 'job_type', 'education_level',
                  'relationship_with_head', 'marital_status', 'location_type']

for col in single_te_cols:
    tr_enc, te_enc = oof_target_encode(X[col], y, X_test[col], smoothing=30)
    X[col + '_te']      = tr_enc
    X_test[col + '_te'] = te_enc

# ── 2-way TE on meaningful pairs ──────────────────────────────────────────
pair_te = [
    ('country',          'job_type'),
    ('country',          'education_level'),
    ('country',          'location_type'),
    ('country',          'cellphone_access'),
    ('job_type',         'education_level'),
    ('job_type',         'gender_of_respondent'),
    ('cellphone_access', 'job_type'),
]

for c1, c2 in pair_te:
    combined_tr = X[c1].astype(str) + '|' + X[c2].astype(str)
    combined_te = X_test[c1].astype(str) + '|' + X_test[c2].astype(str)
    tr_enc, te_enc = oof_target_encode(combined_tr, y, combined_te, smoothing=50)
    name = c1 + '_' + c2 + '_te'
    X[name]      = tr_enc
    X_test[name] = te_enc

FEATURE_COLS = list(X.columns)
print('Features after OOF TE:', len(FEATURE_COLS))
print('New TE columns :', [c for c in FEATURE_COLS if c.endswith('_te')])

Features after OOF TE: 32
New TE columns : ['country_te', 'job_type_te', 'education_level_te', 'relationship_with_head_te', 'marital_status_te', 'location_type_te', 'country_job_type_te', 'country_education_level_te', 'country_location_type_te', 'country_cellphone_access_te', 'job_type_education_level_te', 'job_type_gender_of_respondent_te', 'cellphone_access_job_type_te']


In [17]:
# ══════════════════════════════════════════════════════════════════════════
#  STACKING PIPELINE — Level-1 Base Models (12 diverse learners)
#  Upgrade: folds are stratified on (country, target) so every fold keeps
#  the same country × class distribution → per-country tuning is reliable.
# ══════════════════════════════════════════════════════════════════════════
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# ── Combined (country × target) stratification ────────────────────────────
N_SPLITS = 10
SEED     = 42
strat_key = X['country'].astype(int).values * 2 + y.values  # 2 * K classes
skf       = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds     = list(skf.split(X, strat_key))

# ── Pre-build a scaled copy for distance / linear / NN models ─────────────
scaler        = StandardScaler()
X_scaled      = pd.DataFrame(scaler.fit_transform(X),      columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test),     columns=X.columns)

cat_idx = [list(X.columns).index(c) for c in CAT_COLS if c in list(X.columns)]

# ── Base-model registry ───────────────────────────────────────────────────
def _lgbm_a(seed):
    return LGBMClassifier(n_estimators=5000, learning_rate=0.01, num_leaves=63,
                          min_child_samples=30, feature_fraction=0.80,
                          bagging_fraction=0.80, bagging_freq=5,
                          reg_alpha=0.10, reg_lambda=1.00,
                          random_state=seed, n_jobs=-1, verbose=-1)

def _lgbm_b(seed):
    return LGBMClassifier(n_estimators=5000, learning_rate=0.02, num_leaves=127,
                          min_child_samples=20, feature_fraction=0.70,
                          boosting_type='goss', reg_alpha=0.20, reg_lambda=2.0,
                          random_state=seed, n_jobs=-1, verbose=-1)

def _xgb_a(seed):
    return XGBClassifier(n_estimators=5000, learning_rate=0.01, max_depth=5,
                         min_child_weight=3, subsample=0.80,
                         colsample_bytree=0.80, gamma=0.10,
                         reg_alpha=0.10, reg_lambda=2.00, eval_metric='error',
                         early_stopping_rounds=100, random_state=seed, n_jobs=-1)

def _xgb_b(seed):
    return XGBClassifier(n_estimators=5000, learning_rate=0.005, max_depth=7,
                         min_child_weight=5, subsample=0.70,
                         colsample_bytree=0.70, gamma=0.30,
                         reg_alpha=0.50, reg_lambda=3.00, eval_metric='logloss',
                         early_stopping_rounds=120, random_state=seed, n_jobs=-1)

def _cat_a(seed):
    return CatBoostClassifier(iterations=5000, learning_rate=0.01, depth=6,
                              l2_leaf_reg=5.0, eval_metric='Accuracy',
                              od_type='Iter', od_wait=100, task_type='CPU',
                              random_seed=seed, verbose=False)

def _cat_b(seed):
    return CatBoostClassifier(iterations=5000, learning_rate=0.02, depth=8,
                              l2_leaf_reg=8.0, bootstrap_type='Bernoulli',
                              subsample=0.85, eval_metric='Logloss',
                              od_type='Iter', od_wait=120, task_type='CPU',
                              random_seed=seed, verbose=False)

def _hgb(seed):
    return HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.03,
                                          max_depth=8, min_samples_leaf=30,
                                          l2_regularization=1.0,
                                          early_stopping=True, validation_fraction=0.1,
                                          random_state=seed)

def _rf(seed):
    return RandomForestClassifier(n_estimators=600, max_depth=14,
                                  min_samples_leaf=10, max_features='sqrt',
                                  n_jobs=-1, random_state=seed)

def _et(seed):
    return ExtraTreesClassifier(n_estimators=800, max_depth=16,
                                min_samples_leaf=8, max_features='sqrt',
                                n_jobs=-1, random_state=seed)

def _logreg(seed):
    return LogisticRegression(C=0.5, max_iter=2000, solver='liblinear',
                              random_state=seed)

def _knn(seed):
    return KNeighborsClassifier(n_neighbors=75, weights='distance', n_jobs=-1)

def _mlp(seed):
    return MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                         alpha=1e-3, learning_rate_init=1e-3, max_iter=80,
                         early_stopping=True, validation_fraction=0.1,
                         random_state=seed)

BASE_MODELS = [
    ('lgbm_a',   _lgbm_a,   False, 'lgbm'),
    ('lgbm_b',   _lgbm_b,   False, 'lgbm'),
    ('xgb_a',    _xgb_a,    False, 'xgb'),
    ('xgb_b',    _xgb_b,    False, 'xgb'),
    ('cat_a',    _cat_a,    False, 'cat'),
    ('cat_b',    _cat_b,    False, 'cat'),
    ('hgb',      _hgb,      False, None),
    ('rf',       _rf,       False, None),
    ('et',       _et,       False, None),
    ('logreg',   _logreg,   True,  None),
    ('knn',      _knn,      True,  None),
    ('mlp',      _mlp,      True,  None),
]
print('Level-1 base models :', len(BASE_MODELS))
print([m[0] for m in BASE_MODELS])
print('Fold stratification : country x target  ({} classes)'.format(len(set(strat_key))))

Level-1 base models : 12
['lgbm_a', 'lgbm_b', 'xgb_a', 'xgb_b', 'cat_a', 'cat_b', 'hgb', 'rf', 'et', 'logreg', 'knn', 'mlp']
Fold stratification : country x target  (8 classes)


In [18]:
# ══════════════════════════════════════════════════════════════════════════
#  Train every base model with the SHARED 10-fold split.
#  Produces:
#    OOF[name]       – out-of-fold probabilities (len = len(X))
#    TEST_PRED[name] – fold-averaged test probabilities (len = len(X_test))
# ══════════════════════════════════════════════════════════════════════════
OOF       = {}
TEST_PRED = {}

def _fit_one(name, builder, kind, X_tr, y_tr, X_val, y_val):
    mdl = builder(SEED)
    if kind == 'lgbm':
        if LGBM_NEW_API:
            mdl.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                    categorical_feature=CAT_COLS,
                    callbacks=[early_stopping(100, verbose=False)])
        else:
            mdl.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                    categorical_feature=CAT_COLS,
                    early_stopping_rounds=100, verbose=False)
    elif kind == 'xgb':
        mdl.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    elif kind == 'cat':
        mdl.fit(X_tr, y_tr, cat_features=cat_idx,
                eval_set=(X_val, y_val), use_best_model=True, verbose=False)
    else:
        mdl.fit(X_tr, y_tr)
    return mdl


for name, builder, use_scaled, kind in BASE_MODELS:
    src_X      = X_scaled      if use_scaled else X
    src_X_test = X_test_scaled if use_scaled else X_test

    oof_p  = np.zeros(len(src_X))
    test_p = np.zeros(len(src_X_test))

    t0 = time.time()
    for fold, (tr_idx, val_idx) in enumerate(folds):
        X_tr, X_val = src_X.iloc[tr_idx], src_X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx],     y.iloc[val_idx]

        mdl = _fit_one(name, builder, kind, X_tr, y_tr, X_val, y_val)

        oof_p[val_idx] = mdl.predict_proba(X_val)[:, 1]
        test_p        += mdl.predict_proba(src_X_test)[:, 1] / N_SPLITS

    OOF[name]       = oof_p
    TEST_PRED[name] = test_p
    mae = 1 - accuracy_score(y, (oof_p > 0.5).astype(int))
    print('  {:8s}  OOF MAE = {:.6f}   ({:.1f}s)'.format(
        name, mae, time.time() - t0))

# ── Ranking ────────────────────────────────────────────────────────────────
print('\n=== Level-1 OOF MAE (sorted) ===')
ranked = sorted(((1 - accuracy_score(y, (OOF[n] > 0.5).astype(int))), n)
                for n in OOF)
for mae, n in ranked:
    print('  {:8s} : {:.6f}'.format(n, mae))

  lgbm_a    OOF MAE = 0.113076   (117.2s)
  lgbm_b    OOF MAE = 0.113756   (29.1s)
  xgb_a     OOF MAE = 0.111546   (20.8s)
  xgb_b     OOF MAE = 0.113119   (73.3s)
  cat_a     OOF MAE = 0.112141   (81.7s)
  cat_b     OOF MAE = 0.111631   (360.6s)
  hgb       OOF MAE = 0.113671   (22.0s)
  rf        OOF MAE = 0.112821   (157.8s)
  et        OOF MAE = 0.112736   (137.7s)
  logreg    OOF MAE = 0.111843   (5.2s)
  knn       OOF MAE = 0.118092   (35.4s)
  mlp       OOF MAE = 0.112608   (28.7s)

=== Level-1 OOF MAE (sorted) ===
  xgb_a    : 0.111546
  cat_b    : 0.111631
  logreg   : 0.111843
  cat_a    : 0.112141
  mlp      : 0.112608
  et       : 0.112736
  rf       : 0.112821
  lgbm_a   : 0.113076
  xgb_b    : 0.113119
  hgb      : 0.113671
  lgbm_b   : 0.113756
  knn      : 0.118092


In [19]:
# ══════════════════════════════════════════════════════════════════════════
#  LEVEL-2 META-LEARNERS — trained ONLY on OOF predictions (no leakage).
#  Upgrade: meta-features now include country one-hots + country_te, so the
#  stacker can weight base models differently per country.
# ══════════════════════════════════════════════════════════════════════════
from scipy.optimize import minimize

names     = [n for n, *_ in BASE_MODELS]
oof_base  = np.column_stack([OOF[n]       for n in names])
test_base = np.column_stack([TEST_PRED[n] for n in names])

# ── Country context features for the meta-learner ────────────────────────
country_train = X['country'].astype(int).values
country_test  = X_test['country'].astype(int).values
unique_countries = sorted(set(country_train))

def _onehot(country_arr, categories):
    M = np.zeros((len(country_arr), len(categories)), dtype=np.float32)
    idx_map = {c: i for i, c in enumerate(categories)}
    for i, c in enumerate(country_arr):
        M[i, idx_map[c]] = 1.0
    return M

ctry_oh_tr = _onehot(country_train, unique_countries)
ctry_oh_te = _onehot(country_test,  unique_countries)

# Cross of country one-hot with each base prediction → per-country weighting
ctry_cross_tr = np.column_stack(
    [oof_base[:, j:j+1] * ctry_oh_tr for j in range(oof_base.shape[1])]
).reshape(len(X), -1)
ctry_cross_te = np.column_stack(
    [test_base[:, j:j+1] * ctry_oh_te for j in range(test_base.shape[1])]
).reshape(len(X_test), -1)

oof_mat  = np.hstack([oof_base,  ctry_oh_tr, ctry_cross_tr,
                      X[['country_te']].values])
test_mat = np.hstack([test_base, ctry_oh_te, ctry_cross_te,
                      X_test[['country_te']].values])

# ── 1. Logistic-Regression meta ───────────────────────────────────────────
lr_oof = np.zeros(len(X))
for tr_idx, val_idx in folds:
    meta = LogisticRegression(C=0.5, max_iter=3000, solver='lbfgs')
    meta.fit(oof_mat[tr_idx], y.iloc[tr_idx])
    lr_oof[val_idx] = meta.predict_proba(oof_mat[val_idx])[:, 1]

meta_full = LogisticRegression(C=0.5, max_iter=3000, solver='lbfgs')
meta_full.fit(oof_mat, y)
lr_test = meta_full.predict_proba(test_mat)[:, 1]
lr_mae  = 1 - accuracy_score(y, (lr_oof > 0.5).astype(int))
print('Meta LogReg (country-aware)   OOF MAE :', round(lr_mae, 6))

# ── 2. LightGBM meta (non-linear, sees base preds + country) ─────────────
lgb_meta_in_tr = np.hstack([oof_base,  country_train.reshape(-1, 1),
                            X[['country_te']].values])
lgb_meta_in_te = np.hstack([test_base, country_test.reshape(-1, 1),
                            X_test[['country_te']].values])

lgb_meta_oof = np.zeros(len(X))
for tr_idx, val_idx in folds:
    m = LGBMClassifier(n_estimators=500, learning_rate=0.02, num_leaves=15,
                       min_child_samples=50, reg_lambda=1.0,
                       random_state=SEED, verbose=-1)
    m.fit(lgb_meta_in_tr[tr_idx], y.iloc[tr_idx])
    lgb_meta_oof[val_idx] = m.predict_proba(lgb_meta_in_tr[val_idx])[:, 1]

m_full = LGBMClassifier(n_estimators=500, learning_rate=0.02, num_leaves=15,
                        min_child_samples=50, reg_lambda=1.0,
                        random_state=SEED, verbose=-1)
m_full.fit(lgb_meta_in_tr, y)
lgb_meta_test = m_full.predict_proba(lgb_meta_in_te)[:, 1]
lgb_meta_mae  = 1 - accuracy_score(y, (lgb_meta_oof > 0.5).astype(int))
print('Meta LightGBM (country-aware) OOF MAE :', round(lgb_meta_mae, 6))

# ── 3. Convex blend on base preds (no country) — keeps a linear anchor ───
def neg_acc(w, P, y_):
    w = np.clip(w, 0, None)
    if w.sum() == 0:
        return 1.0
    w = w / w.sum()
    return -accuracy_score(y_, (P @ w > 0.5).astype(int))

best_w, best_score = None, 1.0
rng = np.random.RandomState(SEED)
for _ in range(3000):
    w = rng.dirichlet(np.ones(len(names)))
    s = neg_acc(w, oof_base, y.values)
    if s < best_score:
        best_score, best_w = s, w
res = minimize(neg_acc, best_w, args=(oof_base, y.values),
               method='SLSQP',
               bounds=[(0, 1)] * len(names),
               constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})
if -res.fun > -best_score:
    best_w = np.clip(res.x, 0, None); best_w /= best_w.sum()

blend_oof  = oof_base  @ best_w
blend_test = test_base @ best_w
blend_mae  = 1 - accuracy_score(y, (blend_oof > 0.5).astype(int))
print('Convex blend                  OOF MAE :', round(blend_mae, 6))
print('  Weights :')
for n, w in sorted(zip(names, best_w), key=lambda kv: -kv[1]):
    print('    {:8s} : {:.4f}'.format(n, w))

# ── 4. Per-country convex blend (NEW — weights fit inside each country) ──
country_weights = {}
pc_oof  = np.zeros(len(X))
pc_test = np.zeros(len(X_test))
for c in unique_countries:
    m_tr = country_train == c
    m_te = country_test  == c
    if m_tr.sum() < 200:
        country_weights[c] = best_w
    else:
        bw, bs = best_w, 1.0
        for _ in range(1500):
            w = rng.dirichlet(np.ones(len(names)))
            s = neg_acc(w, oof_base[m_tr], y.values[m_tr])
            if s < bs:
                bs, bw = s, w
        r = minimize(neg_acc, bw, args=(oof_base[m_tr], y.values[m_tr]),
                     method='SLSQP', bounds=[(0, 1)] * len(names),
                     constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})
        if -r.fun > -bs:
            bw = np.clip(r.x, 0, None); bw /= bw.sum()
        # Shrink toward the global weights — Bayesian taste to kill noise
        n_c = m_tr.sum()
        alpha = n_c / (n_c + 2000)          # larger country → trust itself more
        country_weights[c] = alpha * bw + (1 - alpha) * best_w

    pc_oof[m_tr] = oof_base[m_tr]  @ country_weights[c]
    pc_test[m_te] = test_base[m_te] @ country_weights[c]

pc_mae = 1 - accuracy_score(y, (pc_oof > 0.5).astype(int))
print('Per-country convex blend      OOF MAE :', round(pc_mae, 6))

# ── 5. Final meta-blend = mean of the 4 meta predictors ──────────────────
final_oof  = (lr_oof  + lgb_meta_oof  + blend_oof  + pc_oof)  / 4.0
final_test = (lr_test + lgb_meta_test + blend_test + pc_test) / 4.0
final_mae  = 1 - accuracy_score(y, (final_oof > 0.5).astype(int))
print('\n>>> FINAL meta-blend OOF MAE :', round(final_mae, 6))

Meta LogReg (country-aware)   OOF MAE : 0.112268
Meta LightGBM (country-aware) OOF MAE : 0.11061
Convex blend                  OOF MAE : 0.111078
  Weights :
    cat_b    : 0.2672
    mlp      : 0.2060
    rf       : 0.1598
    hgb      : 0.1002
    lgbm_a   : 0.0766
    et       : 0.0620
    cat_a    : 0.0522
    logreg   : 0.0443
    knn      : 0.0113
    xgb_b    : 0.0107
    lgbm_b   : 0.0080
    xgb_a    : 0.0018
Per-country convex blend      OOF MAE : 0.111163

>>> FINAL meta-blend OOF MAE : 0.111546


In [20]:
# ══════════════════════════════════════════════════════════════════════════
#  POST-PROCESSING — threshold tuning done on OOF only.
#  Upgrades:
#    * Bayesian-smoothed per-country thresholds
#      α = n_c / (n_c + 500)  — small countries shrink hard to global
#    * Nested-CV honest estimate of per-country threshold MAE
#    * Percentile scaling that respects per-country rates
# ══════════════════════════════════════════════════════════════════════════
unique_countries = sorted(set(country_train))

# ── A. Global threshold ───────────────────────────────────────────────────
best_t, best_mae = 0.5, 1.0
for t in np.arange(0.20, 0.80, 0.001):
    m = 1 - accuracy_score(y, (final_oof > t).astype(int))
    if m < best_mae:
        best_mae, best_t = m, t
print('A. Global threshold        : t={:.3f}  OOF MAE={:.6f}'.format(best_t, best_mae))

# ── B. Bayesian-smoothed per-country threshold ───────────────────────────
def _best_thr(mask):
    bt, bm = best_t, 1.0
    for t in np.arange(0.10, 0.90, 0.002):
        mm = 1 - accuracy_score(y[mask], (final_oof[mask] > t).astype(int))
        if mm < bm:
            bm, bt = mm, t
    return bt

country_thr = {}
for c in unique_countries:
    mask  = country_train == c
    raw_t = _best_thr(mask)
    n_c   = int(mask.sum())
    alpha = n_c / (n_c + 500)              # stronger shrinkage than 50/50
    country_thr[c] = alpha * raw_t + (1 - alpha) * best_t
    print('   country={}  n={:5d}  raw_t={:.3f}  smoothed_t={:.3f}  α={:.3f}'.format(
        c, n_c, raw_t, country_thr[c], alpha))

oof_pred_country = np.zeros(len(X), dtype=int)
for c in unique_countries:
    mask = country_train == c
    oof_pred_country[mask] = (final_oof[mask] > country_thr[c]).astype(int)
country_mae = 1 - accuracy_score(y, oof_pred_country)
print('B. Per-country thresh (BS) : OOF MAE={:.6f}'.format(country_mae))

# ── B'. Nested-CV honest estimate (train thr on 9 folds, score on 1) ─────
nested_oof = np.zeros(len(X), dtype=int)
for tr_idx, val_idx in folds:
    y_tr = y.iloc[tr_idx].values
    oof_tr = final_oof[tr_idx]
    c_tr   = country_train[tr_idx]
    # Global threshold from training folds
    bt, bm = 0.5, 1.0
    for t in np.arange(0.20, 0.80, 0.002):
        mm = 1 - accuracy_score(y_tr, (oof_tr > t).astype(int))
        if mm < bm:
            bm, bt = mm, t
    # Per-country thresholds on training folds
    thr_map = {}
    for c in unique_countries:
        mc = c_tr == c
        if mc.sum() < 50:
            thr_map[c] = bt
            continue
        bt_c, bm_c = bt, 1.0
        for t in np.arange(0.10, 0.90, 0.004):
            mm = 1 - accuracy_score(y_tr[mc], (oof_tr[mc] > t).astype(int))
            if mm < bm_c:
                bm_c, bt_c = mm, t
        n_c = mc.sum()
        a   = n_c / (n_c + 500)
        thr_map[c] = a * bt_c + (1 - a) * bt
    # Apply to validation fold
    for c in unique_countries:
        mv = country_train[val_idx] == c
        nested_oof[val_idx[mv]] = (final_oof[val_idx[mv]] > thr_map[c]).astype(int)
nested_mae = 1 - accuracy_score(y, nested_oof)
print("B'. Nested-CV honest MAE    : OOF MAE={:.6f}".format(nested_mae))

# ── C. Global percentile match ───────────────────────────────────────────
overall_pos_rate = y.mean()
oof_pred_pct = np.zeros(len(X), dtype=int)
oof_pred_pct[np.argsort(-final_oof)[:int(round(len(X) * overall_pos_rate))]] = 1
pct_mae = 1 - accuracy_score(y, oof_pred_pct)
print('C. Global percentile       : OOF MAE={:.6f}'.format(pct_mae))

# ── D. Per-country percentile ────────────────────────────────────────────
country_pos_rates = {c: y[country_train == c].mean() for c in unique_countries}
oof_pred_pct_c = np.zeros(len(X), dtype=int)
for c in unique_countries:
    mask = country_train == c
    n_pos = int(round(mask.sum() * country_pos_rates[c]))
    if n_pos == 0:
        continue
    idx = np.where(mask)[0]
    oof_pred_pct_c[idx[np.argsort(-final_oof[mask])[:n_pos]]] = 1
pct_country_mae = 1 - accuracy_score(y, oof_pred_pct_c)
print('D. Per-country percentile  : OOF MAE={:.6f}'.format(pct_country_mae))

# ══════════════════════════════════════════════════════════════════════════
#  GENERATE SUBMISSIONS
# ══════════════════════════════════════════════════════════════════════════
sub_A = pd.DataFrame({'uniqueid': test_ids,
                      'bank_account': (final_test > best_t).astype(int)})
sub_A.to_csv('submission_stack_global.csv', index=False)

preds_B = np.zeros(len(X_test), dtype=int)
for c in unique_countries:
    mask = country_test == c
    preds_B[mask] = (final_test[mask] > country_thr[c]).astype(int)
sub_B = pd.DataFrame({'uniqueid': test_ids, 'bank_account': preds_B})
sub_B.to_csv('submission_stack_country_threshold.csv', index=False)

preds_C = np.zeros(len(X_test), dtype=int)
preds_C[np.argsort(-final_test)[:int(round(len(X_test) * overall_pos_rate))]] = 1
sub_C = pd.DataFrame({'uniqueid': test_ids, 'bank_account': preds_C})
sub_C.to_csv('submission_stack_percentile.csv', index=False)

preds_D = np.zeros(len(X_test), dtype=int)
for c in unique_countries:
    mask = country_test == c
    n_pos = int(round(mask.sum() * country_pos_rates[c]))
    if n_pos == 0:
        continue
    idx = np.where(mask)[0]
    preds_D[idx[np.argsort(-final_test[mask])[:n_pos]]] = 1
sub_D = pd.DataFrame({'uniqueid': test_ids, 'bank_account': preds_D})
sub_D.to_csv('submission_stack_percentile_country.csv', index=False)

print('\n=== STACKING v2 — submissions saved ===')
print('  A. Global threshold        : {:.6f}  → submission_stack_global.csv'.format(best_mae))
print('  B. Country threshold (BS)  : {:.6f}  → submission_stack_country_threshold.csv'.format(country_mae))
print("  B'. Nested-CV honest MAE   : {:.6f}".format(nested_mae))
print('  C. Global percentile       : {:.6f}  → submission_stack_percentile.csv'.format(pct_mae))
print('  D. Per-country percentile  : {:.6f}  → submission_stack_percentile_country.csv'.format(pct_country_mae))
for tag, sub in [('A', sub_A), ('B', sub_B), ('C', sub_C), ('D', sub_D)]:
    s = int(sub['bank_account'].sum())
    print('  {} positives : {:5d}/{}  ({:.4f})'.format(tag, s, len(sub), s / len(sub)))
print('  train positive rate : {:.4f}'.format(overall_pos_rate))

A. Global threshold        : t=0.487  OOF MAE=0.111121
   country=0  n= 6068  raw_t=0.524  smoothed_t=0.521  α=0.924
   country=1  n= 8735  raw_t=0.412  smoothed_t=0.416  α=0.946
   country=2  n= 6620  raw_t=0.506  smoothed_t=0.505  α=0.930
   country=3  n= 2101  raw_t=0.416  smoothed_t=0.430  α=0.808
B. Per-country thresh (BS) : OOF MAE=0.110525
B'. Nested-CV honest MAE    : OOF MAE=0.110993
C. Global percentile       : OOF MAE=0.125404
D. Per-country percentile  : OOF MAE=0.127444

=== STACKING v2 — submissions saved ===
  A. Global threshold        : 0.111121  → submission_stack_global.csv
  B. Country threshold (BS)  : 0.110525  → submission_stack_country_threshold.csv
  B'. Nested-CV honest MAE   : 0.110993
  C. Global percentile       : 0.125404  → submission_stack_percentile.csv
  D. Per-country percentile  : 0.127444  → submission_stack_percentile_country.csv
  A positives :   772/10086  (0.0765)
  B positives :   774/10086  (0.0767)
  C positives :  1420/10086  (0.1408)
  D po

In [21]:
# ══════════════════════════════════════════════════════════════════════════
#  PSEUDO-LABELING ROUND — add confident test rows to train, retrain the 3
#  best base models (LGBM_A, XGB_A, CatBoost_A), re-blend, re-threshold.
#  Safety rails:
#    * Only use test rows with final_test > HI or final_test < LO
#    * Original-row class balance is preserved (no runaway positive drift)
#    * Still uses the SAME country×target stratified folds for CV
# ══════════════════════════════════════════════════════════════════════════
HI, LO = 0.90, 0.03
mask_hi = final_test > HI
mask_lo = final_test < LO
print('Pseudo-labels: {} confident positives, {} confident negatives ({:.2%} of test)'
      .format(int(mask_hi.sum()), int(mask_lo.sum()),
              (mask_hi.sum() + mask_lo.sum()) / len(X_test)))

pl_idx = np.where(mask_hi | mask_lo)[0]
X_pl   = X_test.iloc[pl_idx].copy()
y_pl   = pd.Series(mask_hi[pl_idx].astype(int), index=X_pl.index)

X_aug = pd.concat([X, X_pl], ignore_index=True)
y_aug = pd.concat([y, y_pl], ignore_index=True)

# Re-stratify on country×target for the augmented set
strat_aug = X_aug['country'].astype(int).values * 2 + y_aug.values
folds_aug = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True,
                                 random_state=SEED).split(X_aug, strat_aug))

def _cv_predict(builder, kind, X_all, y_all, X_te, folds_):
    oof_  = np.zeros(len(X_all))
    test_ = np.zeros(len(X_te))
    for tr_idx, val_idx in folds_:
        X_tr, X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
        y_tr, y_val = y_all.iloc[tr_idx], y_all.iloc[val_idx]
        mdl = _fit_one('', builder, kind, X_tr, y_tr, X_val, y_val)
        oof_[val_idx] = mdl.predict_proba(X_val)[:, 1]
        test_        += mdl.predict_proba(X_te)[:, 1] / len(folds_)
    return oof_, test_

pl_models = [('lgbm_a', _lgbm_a, 'lgbm'),
             ('xgb_a',  _xgb_a,  'xgb'),
             ('cat_a',  _cat_a,  'cat')]

pl_oof_parts  = []
pl_test_parts = []
for nm, b, k in pl_models:
    t0 = time.time()
    oof_a, test_a = _cv_predict(b, k, X_aug, y_aug, X_test, folds_aug)
    print('  PL {:8s}  trained in {:.1f}s'.format(nm, time.time() - t0))
    # Slice OOF back to the ORIGINAL train rows (first len(X) entries)
    pl_oof_parts.append(oof_a[:len(X)])
    pl_test_parts.append(test_a)

pl_oof_avg  = np.mean(pl_oof_parts,  axis=0)
pl_test_avg = np.mean(pl_test_parts, axis=0)

# Blend pseudo-labeled predictions with the stacked final predictions (60/40)
W_PL = 0.40
pl_final_oof  = (1 - W_PL) * final_oof  + W_PL * pl_oof_avg
pl_final_test = (1 - W_PL) * final_test + W_PL * pl_test_avg

pl_mae = 1 - accuracy_score(y, (pl_final_oof > 0.5).astype(int))
print('\nPseudo-label blended OOF MAE @0.5 : {:.6f}'.format(pl_mae))

# Refit per-country Bayesian-smoothed thresholds on the new OOF
best_t_E, best_mae_E = 0.5, 1.0
for t in np.arange(0.20, 0.80, 0.001):
    m = 1 - accuracy_score(y, (pl_final_oof > t).astype(int))
    if m < best_mae_E:
        best_mae_E, best_t_E = m, t

ctry_thr_E = {}
for c in unique_countries:
    mask = country_train == c
    bt, bm = best_t_E, 1.0
    for t in np.arange(0.10, 0.90, 0.002):
        mm = 1 - accuracy_score(y[mask], (pl_final_oof[mask] > t).astype(int))
        if mm < bm:
            bm, bt = mm, t
    n_c = int(mask.sum())
    a   = n_c / (n_c + 500)
    ctry_thr_E[c] = a * bt + (1 - a) * best_t_E

oof_pred_E = np.zeros(len(X), dtype=int)
for c in unique_countries:
    m = country_train == c
    oof_pred_E[m] = (pl_final_oof[m] > ctry_thr_E[c]).astype(int)
mae_E = 1 - accuracy_score(y, oof_pred_E)
print('E. PL + country threshold     : OOF MAE={:.6f}'.format(mae_E))

preds_E = np.zeros(len(X_test), dtype=int)
for c in unique_countries:
    m = country_test == c
    preds_E[m] = (pl_final_test[m] > ctry_thr_E[c]).astype(int)
sub_E = pd.DataFrame({'uniqueid': test_ids, 'bank_account': preds_E})
sub_E.to_csv('submission_stack_pseudo_country.csv', index=False)
print('   → submission_stack_pseudo_country.csv  (positives: {}/{}  {:.4f})'.format(
    int(preds_E.sum()), len(preds_E), preds_E.mean()))

# Final leaderboard (OOF) — submit whichever scores best on Zindi LB
print('\n=== FINAL OOF RANKING ===')
ranking = [
    ('A. global',              best_mae),
    ('B. country thr (BS)',    country_mae),
    ('C. global pct',          pct_mae),
    ('D. per-country pct',     pct_country_mae),
    ('E. pseudo + country',    mae_E),
]
for name, m in sorted(ranking, key=lambda kv: kv[1]):
    print('  {:25s}  {:.6f}'.format(name, m))

Pseudo-labels: 46 confident positives, 1496 confident negatives (15.29% of test)
  PL lgbm_a    trained in 53.4s
  PL xgb_a     trained in 22.7s
  PL cat_a     trained in 108.3s

Pseudo-label blended OOF MAE @0.5 : 0.111716
E. PL + country threshold     : OOF MAE=0.110185
   → submission_stack_pseudo_country.csv  (positives: 808/10086  0.0801)

=== FINAL OOF RANKING ===
  E. pseudo + country        0.110185
  B. country thr (BS)        0.110525
  A. global                  0.111121
  C. global pct              0.125404
  D. per-country pct         0.127444
